Install & Imports

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import shutil

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models

from sklearn.metrics import confusion_matrix

Dataset Paths

In [4]:
base_path = "/content/drive/My Drive/plant-disease-detection/Plant Disease Detection/Wheat"
output_path = "/content/drive/My Drive/plant-disease-detection/Processed Data/Wheat"

Train / Validation / Test Split

In [3]:
test_ratio = 0.15
val_ratio = 0.15

def split_data(plant_path, output_base):
    plant_name = os.path.basename(plant_path)
    output = os.path.join(output_base, plant_name)

    train_dir = os.path.join(output, "Train")
    val_dir = os.path.join(output, "Validation")
    test_dir = os.path.join(output, "Test")

    for d in [train_dir, val_dir, test_dir]:
        os.makedirs(d, exist_ok=True)

    for disease in os.listdir(plant_path):
        disease_path = os.path.join(plant_path, disease)

        if not os.path.isdir(disease_path):
            continue

        files = [f for f in os.listdir(disease_path) if f.lower().endswith((".jpg",".png",".jpeg"))]
        random.shuffle(files)

        test_count = int(len(files) * test_ratio)
        val_count = int(len(files) * val_ratio)

        test_files = files[:test_count]
        val_files = files[test_count:test_count+val_count]
        train_files = files[test_count+val_count:]

        for file_list, target in zip(
            [train_files, val_files, test_files],
            [train_dir, val_dir, test_dir]
        ):
            out_dir = os.path.join(target, disease)
            os.makedirs(out_dir, exist_ok=True)

            for f in file_list:
                shutil.copy(os.path.join(disease_path, f), os.path.join(out_dir, f))

        print(disease, len(train_files), len(val_files), len(test_files))

In [ ]:
split_data(base_path, output_path)

Data Generators + Augmentation

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    os.path.join(output_path, "Wheat/Train"),
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

val_data = test_gen.flow_from_directory(
    os.path.join(output_path, "Wheat/Validation"),
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

test_data = test_gen.flow_from_directory(
    os.path.join(output_path, "Wheat/Test"),
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

Model (VGG16)

In [ ]:
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(512, activation="relu"),
    layers.Dense(train_data.num_classes, activation="softmax")
])

base_model.trainable = False

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Training

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(test_data)
print("Test Accuracy:", test_acc * 100)

Confusion Matrix

In [ ]:
y_pred = model.predict(test_data)
y_pred_classes = np.argmax(y_pred, axis=1)

y_true = test_data.classes

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=test_data.class_indices.keys(),
            yticklabels=test_data.class_indices.keys())
plt.title("Wheat Confusion Matrix")
plt.show()

Save Model

In [ ]:
model.save("/content/drive/My Drive/plant-disease-detection/models/wheat_model.h5")
print("Saved!")

Inference Test

In [ ]:
from tensorflow.keras.preprocessing import image

img_path = "test.jpg"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
print(np.argmax(pred))